In [3]:

import pandas as pd
import os
import wfdb
from scipy.signal import butter, filtfilt

DATA_DIR = "/home/sayantan/Downloads/mit-bih-arrhythmia-database-1.0.0"

# All valid MIT-BIH arrhythmia records
RECORDS = [
    "100","101","102","103","104","105","106","107",
    "108","109","111","112","113","114","115","116",
    "117","118","119","121","122","123","124","200",
    "201","202","203","205","207","208","209","210",
    "212","213","214","215","217","219","220","221",
    "222","223","228","230","231","232","233","234"
]

# R-peak window in seconds
PRE_SEC = 0.2        # 200 ms before R
POST_SEC = 0.4       # 400 ms after R

symbol_to_aami = {
    # Normal (N)
    'N':'N','L':'N','R':'N','e':'N','j':'N',

    # Supraventricular (S)
    'A':'S','a':'S','J':'S','S':'S',

    # Ventricular (V)
    'V':'V','E':'V',

    # Fusion (F)
    'F':'F',
}

aami_to_int = {'N':0, 'S':1, 'V':2, 'F':3}

# FILTER: BANDPASS 0.5–40 Hz
# ==========================================
def bandpass_filter(sig, fs, low=0.5, high=40.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, sig)


# MAIN LOOP — PROCESS ALL RECORDS
all_beats = []
all_labels = []

for rec in RECORDS:
    path = os.path.join(DATA_DIR, rec)

    print(f"Processing record: {rec}")

    # Read ECG & annotation
    record = wfdb.rdrecord(path)
    ann = wfdb.rdann(path, 'atr')

    # Use MLII (channel 0)
    ecg = record.p_signal[:, 0].astype(np.float32)
    fs = record.fs

    # Filter the signal
    ecg_f = bandpass_filter(ecg, fs)

    r_peaks = ann.sample
    symbols = ann.symbol

    # convert seconds to samples
    pre = int(PRE_SEC * fs)
    post = int(POST_SEC * fs)
    beat_len = pre + post

    # iterate all R-peaks
    for r, sym in zip(r_peaks, symbols):

        # skip unknown beat types
        if sym not in symbol_to_aami:
            continue

        start = r - pre
        end = r + post

        # skip out-of-range beats
        if start < 0 or end >= len(ecg_f):
            continue

        beat = ecg_f[start:end]

        if len(beat) != beat_len:
            continue

        # map to AAMI → int
        aami = symbol_to_aami[sym]
        label = aami_to_int[aami]

        all_beats.append(beat)
        all_labels.append(label)

print("\nFinished preprocessing.")

all_beats = np.array(all_beats)
all_labels = np.array(all_labels)

print("Beats shape :", all_beats.shape)
print("Labels shape:", all_labels.shape)

for aami_class, idx in aami_to_int.items():
    print(f"{aami_class}: {(all_labels == idx).sum()} beats")

# ==========================================
# SAVE PROCESSED DATA
np.save("X_beats.npy", all_beats)
np.save("y_labels.npy", all_labels)

print("\nSaved: X_beats.npy, y_labels.npy")



Processing record: 100
Processing record: 101
Processing record: 102
Processing record: 103
Processing record: 104
Processing record: 105
Processing record: 106
Processing record: 107
Processing record: 108
Processing record: 109
Processing record: 111
Processing record: 112
Processing record: 113
Processing record: 114
Processing record: 115
Processing record: 116
Processing record: 117
Processing record: 118
Processing record: 119
Processing record: 121
Processing record: 122
Processing record: 123
Processing record: 124
Processing record: 200
Processing record: 201
Processing record: 202
Processing record: 203
Processing record: 205
Processing record: 207
Processing record: 208
Processing record: 209
Processing record: 210
Processing record: 212
Processing record: 213
Processing record: 214
Processing record: 215
Processing record: 217
Processing record: 219
Processing record: 220
Processing record: 221
Processing record: 222
Processing record: 223
Processing record: 228
Processing 

In [2]:
import numpy as np

In [4]:
y = np.load("y_labels.npy")

print("Unique labels:", np.unique(y))
print("Counts:", [(i, (y==i).sum()) for i in range(4)])

Unique labels: [0 1 2 3]
Counts: [(0, np.int64(90600)), (1, np.int64(2781)), (2, np.int64(7235)), (3, np.int64(802))]
